# OmniField-JEPA v2 — soft-positioned latents, target-anchored geometry, collapse diagnostics

This rewrites the JEPA training to fix the structural mismatch between OmniField's learnable-query
encoder and i-JEPA's position-indexed token assumption.

## What changed vs. the previous version

1. **Soft-positioned latents (v3 — explicit learnable anchors).** Each latent token has an
   explicit `latent_pos ∈ ℝ²` parameter, initialized on a uniform grid in [-1, 1]² with
   small jitter. (v2 derived this from cross-attention weights, but that's near-uniform at
   init → all latents got the same centroid → collapse before warmup ended. v3 sidesteps the
   chicken-and-egg by decoupling position from attention, ViT-style.)
2. **Position-aware `CoordReadout`.** Context tokens are now `proj_g(g) + proj_pos(γ(soft_pos))`,
   matching the target tokens' `proj_query(γ(target_coord)) + mask_token` so cross-attention
   between them has a meaningful spatial reference frame.
3. **Target geometry from the target encoder.** Query coordinates for the JEPA loss are
   *the target encoder's own stage-1 soft positions* — sample-adaptive anchors guaranteed to lie
   inside the input distribution. Removes the M_PRED chunk sampler from the JEPA side.
4. **Soft-position variance regularizer** as an explicit anti-collapse signal — degenerate
   attention ⇒ identical soft positions ⇒ loss penalty.
5. **Verbose per-step collapse diagnostics.** Every `LOG_EVERY` steps the loop prints norms,
   batch-axis std of the predictor output (the clearest collapse signal), token-axis std of `g`,
   soft-position variance, target-prediction cosine similarity, and `||center||`.

## Carried over

- DINO-style EMA target centering (`target_center`) before LayerNorm.
- EMA momentum `0.999 → 1.0` over the schedule.
- Predictor warmup: `WARMUP_STEPS = 2000` frozen-encoder steps before context_encoder gradients
  flow.
- 20 %-only test-time constraint: test loader exposes the fixed first-K_HALF of the pool and
  empty target/pool tensors so downstream code cannot peek.


In [ ]:
import os, math, ssl, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from einops import rearrange, repeat
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt

ssl._create_default_https_context = ssl._create_unverified_context
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- shared experiment config ----
IMAGE_SIZE  = 32
N_PIX       = IMAGE_SIZE * IMAGE_SIZE          # 1024
FRAC_POOL   = 0.40
K_POOL      = int(round(FRAC_POOL * N_PIX))    # 410
K_HALF      = K_POOL // 2                      # 205 — context size; also = fixed test-time input

CHANNELS    = 3
FOURIER_DIM = 96
POS_DIM     = FOURIER_DIM * 2                  # 192
INPUT_DIM   = CHANNELS + POS_DIM               # 195

LATENT_DIMS = (256, 384, 512)
NUM_LATENTS = (256, 256, 256)
EMBED_DIM   = LATENT_DIMS[-1]                  # 512
PRED_DIM    = 256
PRED_LAYERS = 4
PRED_HEADS  = 8
PRED_HEAD_D = 32

# ---- JEPA-specific config ----
BATCH_SIZE       = 64
EPOCHS           = 30
LR               = 2e-4

EMA_START        = 0.999
EMA_END          = 1.000
WARMUP_STEPS     = 2000          # encoder frozen for the first N steps
CENTER_MOMENTUM  = 0.9           # DINO centering EMA
POS_VAR_WEIGHT   = 0.1           # weight on -var(soft_pos) anti-collapse aux loss
LOG_EVERY        = 50            # verbose diagnostics frequency, in optimizer steps

POOL_SEED   = 12345
CKPT_DIR    = "./checkpoints_jepa"
RESUME      = True
PROBE_RESUME= True

print(f"Device: {DEVICE}")
print(f"K_POOL = {K_POOL}  K_HALF = {K_HALF} (also test-time 20%)  EMBED_DIM = {EMBED_DIM}  PRED_DIM = {PRED_DIM}")
print(f"EMA: {EMA_START} → {EMA_END}  WARMUP_STEPS = {WARMUP_STEPS}  CENTER_MOMENTUM = {CENTER_MOMENTUM}")
print(f"POS_VAR_WEIGHT = {POS_VAR_WEIGHT}  LOG_EVERY = {LOG_EVERY}")


In [ ]:
class JEPAChunkCIFAR10(Dataset):
    """Per-instance fixed 40% pool with single-chunk context sampling.

    train=True : pick a random anchor in the pool, take K_HALF nearest pool members as context.
    train=False: only the fixed first-K_HALF of the pool is exposed (the 20% test-time input);
                 pool fields come back as empty tensors so downstream code cannot peek.
    """
    def __init__(self, base_dataset, image_size=IMAGE_SIZE, frac_pool=FRAC_POOL,
                 seed=POOL_SEED, train=True):
        self.base   = base_dataset
        self.N_pix  = image_size * image_size
        self.K_pool = int(round(frac_pool * self.N_pix))
        self.k_half = self.K_pool // 2
        self.train  = train

        rng = np.random.RandomState(seed)
        self.pool_idx = np.stack(
            [rng.permutation(self.N_pix)[:self.K_pool] for _ in range(len(base_dataset))],
            axis=0,
        ).astype(np.int64)

        ys, xs = torch.meshgrid(
            torch.linspace(-1.0, 1.0, image_size),
            torch.linspace(-1.0, 1.0, image_size),
            indexing='ij',
        )
        self.coords_all = torch.stack([ys, xs], dim=-1).view(self.N_pix, 2)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, label = self.base[idx]
        pix_all    = img.permute(1, 2, 0).reshape(-1, 3)
        pool       = self.pool_idx[idx]
        pool_xy    = self.coords_all[pool]

        if self.train:
            anchor_local = np.random.randint(self.K_pool)
            d2           = (pool_xy - pool_xy[anchor_local]).pow(2).sum(-1).numpy()
            order        = np.argsort(d2, kind="stable")
            ctx_local    = order[:self.k_half]
            ctx_idx      = pool[ctx_local]
            return {
                "ctx_pixels":   pix_all[ctx_idx],
                "ctx_coords":   self.coords_all[ctx_idx],
                "pool_pixels":  pix_all[pool],
                "pool_coords":  self.coords_all[pool],
                "ctx_idx":      torch.from_numpy(ctx_idx),
                "label":        int(label),
            }
        else:
            ctx_idx  = pool[:self.k_half]
            empty_px = torch.empty(0, 3)
            empty_xy = torch.empty(0, 2)
            return {
                "ctx_pixels":   pix_all[ctx_idx],
                "ctx_coords":   self.coords_all[ctx_idx],
                "pool_pixels":  empty_px,
                "pool_coords":  empty_xy,
                "ctx_idx":      torch.from_numpy(ctx_idx),
                "label":        int(label),
            }


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3),
])
cifar_train = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform)
cifar_test  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_ds = JEPAChunkCIFAR10(cifar_train, train=True)
test_ds  = JEPAChunkCIFAR10(cifar_test,  train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

batch = next(iter(train_loader))
print("train batch shapes:")
for k, v in batch.items():
    print(f"  {k:13s} {tuple(v.shape)}  {v.dtype}")


In [ ]:
# ---- OmniField backbone with soft-position extraction ----

class GaussianFourierFeatures(nn.Module):
    def __init__(self, in_features, mapping_size, scale=15.0):
        super().__init__()
        self.register_buffer('B', torch.randn((in_features, mapping_size)) * scale)
    def forward(self, coords):
        proj = coords @ self.B
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)


def get_sinusoidal_embeddings(n, d):
    assert d % 2 == 0
    position = torch.arange(n, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d, 2).float() * -(math.log(10000.0) / d))
    pe = torch.zeros(n, d)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


class PreNorm(nn.Module):
    def __init__(self, dim, fn, context_dim=None):
        super().__init__()
        self.fn   = fn
        self.norm = nn.LayerNorm(dim)
        self.norm_context = nn.LayerNorm(context_dim) if context_dim is not None else None
    def forward(self, x, **kw):
        x = self.norm(x)
        if self.norm_context is not None:
            kw['context'] = self.norm_context(kw['context'])
        return self.fn(x, **kw)


class GEGLU(nn.Module):
    def forward(self, x):
        x, gates = x.chunk(2, dim=-1)
        return x * F.gelu(gates)


class FeedForward(nn.Module):
    def __init__(self, dim, mult=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * mult * 2),
            GEGLU(),
            nn.Linear(dim * mult, dim),
        )
    def forward(self, x):
        return self.net(x)


class Attention(nn.Module):
    """Standard cross/self attention."""
    def __init__(self, query_dim, context_dim=None, heads=8, dim_head=64):
        super().__init__()
        inner = dim_head * heads
        context_dim = context_dim if context_dim is not None else query_dim
        self.scale, self.heads = dim_head ** -0.5, heads
        self.to_q   = nn.Linear(query_dim, inner, bias=False)
        self.to_kv  = nn.Linear(context_dim, inner * 2, bias=False)
        self.to_out = nn.Linear(inner, query_dim)
    def forward(self, x, context=None, mask=None):
        h = self.heads
        q = self.to_q(x)
        context = context if context is not None else x
        k, v = self.to_kv(context).chunk(2, dim=-1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> (b h) n d', h=h), (q, k, v))
        sim  = torch.einsum('b i d, b j d -> b i j', q, k) * self.scale
        attn = sim.softmax(dim=-1)
        out  = torch.einsum('b i j, b j d -> b i d', attn, v)
        return self.to_out(rearrange(out, '(b h) n d -> b n (h d)', h=h))


class CascadedBlock(nn.Module):
    def __init__(self, dim, n_latents, input_dim, cross_heads, cross_dim_head,
                 self_heads, self_dim_head, residual_dim=None):
        super().__init__()
        self.latents       = nn.Parameter(get_sinusoidal_embeddings(n_latents, dim), requires_grad=True)
        self.cross_attn    = PreNorm(dim, Attention(dim, input_dim, heads=cross_heads,
                                                    dim_head=cross_dim_head), context_dim=input_dim)
        self.self_attn     = PreNorm(dim, Attention(dim, heads=self_heads, dim_head=self_dim_head))
        self.residual_proj = nn.Linear(residual_dim, dim) if residual_dim and residual_dim != dim else None
        self.ff            = PreNorm(dim, FeedForward(dim))
    def forward(self, context, residual=None):
        b = context.size(0)
        x = repeat(self.latents, 'n d -> b n d', b=b)
        x = self.cross_attn(x, context=context) + x
        if residual is not None:
            r = self.residual_proj(residual) if self.residual_proj is not None else residual
            x = x + r
        x = self.self_attn(x) + x
        x = self.ff(x) + x
        return x


class JEPAEncoder(nn.Module):
    """OmniField backbone (ICMR cascade + trunk) with *explicit learnable
    2D positions* per latent token.

    Why not derive position from cross-attention weights (the original v2 design)?
    At initialization the cross-attention is near-uniform (small qk logits ⇒ softmax
    is flat), so the attention-weighted-average over input coords is the centroid
    for every latent ⇒ var(soft_pos) = 0 ⇒ JEPA collapses to a per-image constant.
    With explicit learnable anchors initialized on a grid in [-1, 1]², soft_pos
    has high variance from step 0 and the aux variance regularizer is well-behaved.
    """
    def __init__(self, input_dim=INPUT_DIM,
                 latent_dims=LATENT_DIMS, num_latents=NUM_LATENTS,
                 cross_heads=4, cross_dim_head=128,
                 self_heads=8, self_dim_head=128,
                 num_trunk_layers=4):
        super().__init__()
        self.encoder_blocks = nn.ModuleList()
        prev = None
        for d, n in zip(latent_dims, num_latents):
            self.encoder_blocks.append(
                CascadedBlock(d, n, input_dim, cross_heads, cross_dim_head,
                              self_heads, self_dim_head, residual_dim=prev)
            )
            prev = d
        final = latent_dims[-1]
        self.trunk = nn.ModuleList([
            nn.ModuleList([
                PreNorm(final, Attention(final, heads=self_heads, dim_head=self_dim_head)),
                PreNorm(final, FeedForward(final)),
            ]) for _ in range(num_trunk_layers)
        ])
        self.latent_dim = final

        # Learnable 2D anchor per latent token (stage-1 worth — passed to readouts as soft_pos)
        N_lat = num_latents[0]
        gs    = int(math.ceil(math.sqrt(N_lat)))
        yx    = torch.linspace(-1.0, 1.0, gs)
        grid  = torch.stack(torch.meshgrid(yx, yx, indexing="ij"), dim=-1).view(-1, 2)
        if grid.shape[0] < N_lat:
            pad = torch.zeros(N_lat - grid.shape[0], 2)
            grid = torch.cat([grid, pad], dim=0)
        else:
            grid = grid[:N_lat]
        # Small jitter so anchors aren't perfectly on a deterministic grid
        grid = grid + 0.01 * torch.randn_like(grid)
        self.latent_pos = nn.Parameter(grid, requires_grad=True)

    def forward(self, data, input_coords=None):
        """data:         (B, N_in, INPUT_DIM)
           input_coords: unused now (kept for API compatibility with v2); the
                         latent's position is `self.latent_pos`, not derived
                         from attention.
        Returns: (g, soft_pos) where soft_pos is the learnable anchor broadcast
                 to the batch dim.
        """
        residual = None
        for block in self.encoder_blocks:
            residual = block(data, residual=residual)
        for sa, ff in self.trunk:
            residual = sa(residual) + residual
            residual = ff(residual) + residual
        soft_pos = self.latent_pos.unsqueeze(0).expand(data.size(0), -1, -1)
        return residual, soft_pos


class CoordReadout(nn.Module):
    """Per-coordinate readout used as both predictor (trainable) and target_readout
    (EMA of predictor). Context tokens now carry an explicit positional encoding
    γ(soft_pos), sharing the positional frame with the target tokens."""
    def __init__(self, latent_dim=EMBED_DIM, predictor_dim=PRED_DIM,
                 query_in_dim=POS_DIM, num_layers=PRED_LAYERS,
                 heads=PRED_HEADS, dim_head=PRED_HEAD_D):
        super().__init__()
        self.proj_g     = nn.Linear(latent_dim, predictor_dim)
        self.proj_pos   = nn.Linear(query_in_dim, predictor_dim)     # γ(soft_pos) → ctx pos enc
        self.proj_query = nn.Linear(query_in_dim, predictor_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, predictor_dim))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        self.blocks = nn.ModuleList([
            nn.ModuleList([
                PreNorm(predictor_dim, Attention(predictor_dim, heads=heads, dim_head=dim_head)),
                PreNorm(predictor_dim, FeedForward(predictor_dim)),
            ]) for _ in range(num_layers)
        ])
        self.norm     = nn.LayerNorm(predictor_dim)
        self.proj_out = nn.Linear(predictor_dim, latent_dim)

    def forward(self, g, query_coords_enc, soft_pos_enc):
        """g:                (B, N_lat, latent_dim)
           query_coords_enc: (B, N_q,   POS_DIM)   = γ(target_coord)
           soft_pos_enc:     (B, N_lat, POS_DIM)   = γ(soft_pos of g)
        """
        B, N_lat, _ = g.shape
        ctx_tok = self.proj_g(g) + self.proj_pos(soft_pos_enc)            # content + position
        tgt_tok = self.proj_query(query_coords_enc) + self.mask_token     # mask + position
        x = torch.cat([ctx_tok, tgt_tok], dim=1)
        for sa, ff in self.blocks:
            x = sa(x) + x
            x = ff(x) + x
        x = self.norm(x)
        return self.proj_out(x[:, N_lat:])


In [ ]:
import copy

fourier_encoder = GaussianFourierFeatures(2, FOURIER_DIM, scale=15.0).to(DEVICE)
context_encoder = JEPAEncoder().to(DEVICE)
predictor       = CoordReadout(latent_dim=EMBED_DIM, predictor_dim=PRED_DIM).to(DEVICE)

target_encoder  = copy.deepcopy(context_encoder).to(DEVICE)
target_readout  = copy.deepcopy(predictor).to(DEVICE)
for p in target_encoder.parameters(): p.requires_grad_(False)
for p in target_readout.parameters(): p.requires_grad_(False)


# DINO-style EMA centering for the target embedding
class TargetCenter(nn.Module):
    def __init__(self, embed_dim, momentum=CENTER_MOMENTUM):
        super().__init__()
        self.momentum = momentum
        self.register_buffer("center", torch.zeros(1, 1, embed_dim))
    @torch.no_grad()
    def update(self, h):
        batch_center = h.mean(dim=(0, 1), keepdim=True)
        self.center.mul_(self.momentum).add_(batch_center, alpha=1.0 - self.momentum)
    def forward(self, h):
        return h - self.center


target_center = TargetCenter(EMBED_DIM, momentum=CENTER_MOMENTUM).to(DEVICE)

n_enc  = sum(p.numel() for p in context_encoder.parameters())
n_pred = sum(p.numel() for p in predictor.parameters())
print(f"context_encoder: {n_enc/1e6:.2f}M  predictor: {n_pred/1e6:.2f}M")

trainable_params = (
    list(context_encoder.parameters())
    + list(predictor.parameters())
    + list(fourier_encoder.parameters())
)
optimizer = AdamW(trainable_params, lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader))


def encode_input(pixels, coords):
    return torch.cat([pixels, fourier_encoder(coords)], dim=-1)


def encode_query(coords):
    return fourier_encoder(coords)


@torch.no_grad()
def ema_update(target_module, online_module, m):
    for p_q, p_k in zip(online_module.parameters(), target_module.parameters()):
        p_k.data.mul_(m).add_((1.0 - m) * p_q.detach())


def make_momentum_schedule(start, end, total_steps):
    def gen():
        for i in range(total_steps + 1):
            yield start + i * (end - start) / total_steps
    return gen()


In [ ]:
# ---- Checkpoint utilities ----

os.makedirs(CKPT_DIR, exist_ok=True)
JEPA_LAST  = os.path.join(CKPT_DIR, "jepa_last.pt")
JEPA_BEST  = os.path.join(CKPT_DIR, "jepa_best.pt")
PROBE_LAST = os.path.join(CKPT_DIR, "probe_last.pt")
PROBE_BEST = os.path.join(CKPT_DIR, "probe_best.pt")

for _name in os.listdir(CKPT_DIR):
    if _name.startswith(".ckpt_") or _name.endswith(".pt.tmp"):
        try:    os.remove(os.path.join(CKPT_DIR, _name))
        except OSError: pass


def _save_atomic(state, path):
    tmp = path + ".tmp"
    try:
        torch.save(state, tmp)
        os.replace(tmp, path)
    except Exception:
        if os.path.exists(tmp):
            try: os.remove(tmp)
            except OSError: pass
        raise


def save_jepa_ckpt(path, *, epoch, train_loss, best_loss, momentum, history, global_step):
    _save_atomic({
        "epoch":            epoch,
        "context_encoder":  context_encoder.state_dict(),
        "predictor":        predictor.state_dict(),
        "target_encoder":   target_encoder.state_dict(),
        "target_readout":   target_readout.state_dict(),
        "fourier":          fourier_encoder.state_dict(),
        "optimizer":        optimizer.state_dict(),
        "scheduler":        scheduler.state_dict(),
        "target_center":    target_center.state_dict(),
        "train_loss":       train_loss,
        "best_loss":        best_loss,
        "momentum":         momentum,
        "global_step":      global_step,
        "history":          history,
    }, path)


def load_jepa_ckpt(path, map_location=DEVICE):
    s = torch.load(path, map_location=map_location, weights_only=False)
    context_encoder.load_state_dict(s["context_encoder"])
    predictor.load_state_dict(s["predictor"])
    target_encoder.load_state_dict(s["target_encoder"])
    target_readout.load_state_dict(s["target_readout"])
    fourier_encoder.load_state_dict(s["fourier"])
    optimizer.load_state_dict(s["optimizer"])
    scheduler.load_state_dict(s["scheduler"])
    if "target_center" in s:
        target_center.load_state_dict(s["target_center"])
    return s


def save_probe_ckpt(path, *, probe, epoch, train_acc, test_acc, best_acc):
    _save_atomic({
        "epoch": epoch, "probe": probe.state_dict(),
        "train_acc": train_acc, "test_acc": test_acc, "best_acc": best_acc,
    }, path)


def load_probe_ckpt(path, probe, map_location=DEVICE):
    s = torch.load(path, map_location=map_location, weights_only=False)
    probe.load_state_dict(s["probe"])
    return s


for p in (JEPA_BEST, JEPA_LAST, PROBE_BEST, PROBE_LAST):
    if os.path.exists(p):
        s = torch.load(p, map_location="cpu", weights_only=False)
        if "train_loss" in s:
            print(f"  found  {p}  @ epoch {s['epoch']}  train_loss={s['train_loss']:.4f}  m={s.get('momentum', float('nan')):.4f}")
        else:
            print(f"  found  {p}  @ epoch {s['epoch']}  test_acc={s.get('test_acc', float('nan')):.4f}")
    else:
        print(f"  none   {p}")


In [ ]:
# ---- JEPA training step (soft-positioned latents, target-anchored query, full diagnostics) ----

# v3 note: soft_pos is now the encoder's learnable `latent_pos` parameter
# (broadcast across the batch), not the cross-attention-weighted centroid.
# sp_var should be > 0.1 throughout training; if it drops, the encoder is
# pushing latent_pos toward zero variance — meaning the anchors are
# collapsing despite the aux variance regularizer.

def jepa_step(batch, compute_diag=False):
    """Runs one JEPA forward-backward step. Returns (loss_total, diag_dict).
    The loss is `smooth_l1(predictor, sg(LN(target_readout − center))) − λ · var(soft_pos)`."""
    ctx_p  = batch['ctx_pixels'].to(DEVICE)
    ctx_c  = batch['ctx_coords'].to(DEVICE)
    pool_p = batch['pool_pixels'].to(DEVICE)
    pool_c = batch['pool_coords'].to(DEVICE)

    ctx_data  = encode_input(ctx_p,  ctx_c)
    pool_data = encode_input(pool_p, pool_c)

    # ---- Target side ----
    with torch.no_grad():
        g_tgt, soft_pos_tgt = target_encoder(pool_data, None)
        tgt_pos_enc         = encode_query(soft_pos_tgt)          # γ(soft_pos_tgt) — also the query
        h_tgt_raw           = target_readout(g_tgt, tgt_pos_enc, tgt_pos_enc)
        target_center.update(h_tgt_raw)
        h_tgt               = target_center(h_tgt_raw)
        h_tgt               = F.layer_norm(h_tgt, (h_tgt.size(-1),))

    # ---- Context side: predict at the SAME target-derived query coords ----
    g_ctx, soft_pos_ctx = context_encoder(ctx_data, None)
    ctx_pos_enc         = encode_query(soft_pos_ctx)              # γ(soft_pos_ctx) for context tokens
    h_pred              = predictor(g_ctx, tgt_pos_enc, ctx_pos_enc)

    loss_jepa = F.smooth_l1_loss(h_pred, h_tgt)

    # ---- Anti-collapse: encourage soft positions to spread (across latents) ----
    pos_var = soft_pos_ctx.var(dim=1).mean() + soft_pos_tgt.var(dim=1).mean()
    loss = loss_jepa - POS_VAR_WEIGHT * pos_var

    diag = {}
    if compute_diag:
        with torch.no_grad():
            diag["loss"]         = loss.item()
            diag["loss_jepa"]    = loss_jepa.item()
            diag["pos_var"]      = pos_var.item()
            diag["g_ctx_norm"]   = g_ctx.norm(dim=-1).mean().item()
            diag["h_pred_norm"]  = h_pred.norm(dim=-1).mean().item()
            diag["h_tgt_norm"]   = h_tgt_raw.norm(dim=-1).mean().item()
            # CRITICAL collapse signals — should NOT drop near zero
            diag["std_b_hpred"]  = h_pred.std(dim=0).mean().item()   # batch-axis variability
            diag["std_t_gctx"]   = g_ctx.std(dim=1).mean().item()    # token-axis variability
            diag["sp_var_ctx"]   = soft_pos_ctx.var(dim=1).mean().item()
            diag["sp_var_tgt"]   = soft_pos_tgt.var(dim=1).mean().item()
            cos = F.cosine_similarity(h_pred, h_tgt, dim=-1)         # (B, N_q)
            diag["cos_mean"]     = cos.mean().item()
            diag["cos_std"]      = cos.std().item()
            diag["center_norm"]  = target_center.center.norm().item()
    return loss, diag


def _set_encoder_trainable(flag: bool):
    for p in context_encoder.parameters():
        p.requires_grad_(flag)
        if not flag:
            p.grad = None


def _fmt_diag(d, step, epoch, lr, m, warmup_active):
    flag = "W" if warmup_active else " "
    msg = (f"[{flag} ep{epoch:02d} st{step:05d}]  "
           f"L={d['loss']:.4f}  Lj={d['loss_jepa']:.4f}  "
           f"|g|={d['g_ctx_norm']:.2f} |h_p|={d['h_pred_norm']:.2f} |h_t|={d['h_tgt_norm']:.2f}  "
           f"σb(h_p)={d['std_b_hpred']:.3f} σt(g)={d['std_t_gctx']:.3f}  "
           f"svar(c|t)={d['sp_var_ctx']:.3f}|{d['sp_var_tgt']:.3f}  "
           f"cos={d['cos_mean']:.3f}±{d['cos_std']:.3f}  "
           f"|cen|={d['center_norm']:.2f}  lr={lr:.1e} m={m:.4f}")
    return msg


def _collapse_warn(d):
    msgs = []
    if d["std_b_hpred"] < 0.01: msgs.append(f"std_b(h_pred)={d['std_b_hpred']:.4f}<0.01")
    if d["std_t_gctx"]  < 0.05: msgs.append(f"std_t(g_ctx)={d['std_t_gctx']:.4f}<0.05")
    if d["sp_var_ctx"]  < 0.005: msgs.append(f"sp_var_ctx={d['sp_var_ctx']:.4f}<0.005")
    if d["sp_var_tgt"]  < 0.005: msgs.append(f"sp_var_tgt={d['sp_var_tgt']:.4f}<0.005")
    return "  !! COLLAPSE WARNING: " + ", ".join(msgs) if msgs else ""


# ---- Training loop ----
history     = {"train_loss": [], "momentum": [], "diag_steps": [], "diag_log": []}
start_epoch = 1
best_loss   = float("inf")
total_steps = EPOCHS * len(train_loader)
momentum_gen = make_momentum_schedule(EMA_START, EMA_END, total_steps)
global_step  = 0

if RESUME and os.path.exists(JEPA_LAST):
    try:
        s = load_jepa_ckpt(JEPA_LAST)
        start_epoch = s["epoch"] + 1
        history     = s["history"]
        best_loss   = s.get("best_loss", float("inf"))
        global_step = s.get("global_step", (start_epoch - 1) * len(train_loader))
        if os.path.exists(JEPA_BEST):
            b = torch.load(JEPA_BEST, map_location="cpu", weights_only=False)
            best_loss = min(best_loss, b["train_loss"])
        for _ in range(global_step):
            try: next(momentum_gen)
            except StopIteration: break
        print(f"Resumed JEPA from {JEPA_LAST} @ epoch {s['epoch']} step {global_step} (best_loss={best_loss:.4f})")
    except Exception as e:
        print(f"Could not resume ({e}); starting fresh.")
        history     = {"train_loss": [], "momentum": [], "diag_steps": [], "diag_log": []}
        start_epoch = 1
        best_loss   = float("inf")
        global_step = 0
else:
    print(f"Starting JEPA fresh — no resume.")

_set_encoder_trainable(global_step >= WARMUP_STEPS)
if global_step < WARMUP_STEPS:
    print(f"Predictor warmup active: context_encoder frozen for {WARMUP_STEPS - global_step} more steps.")

epoch = start_epoch - 1
m = EMA_START
try:
    for epoch in range(start_epoch, EPOCHS + 1):
        context_encoder.train(); predictor.train(); fourier_encoder.train()
        epoch_loss, steps = 0.0, 0
        t_epoch = time.time()

        for batch in train_loader:
            warmup_active = global_step < WARMUP_STEPS
            do_diag       = (global_step % LOG_EVERY == 0)

            loss, diag = jepa_step(batch, compute_diag=do_diag)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            optimizer.step()
            scheduler.step()

            try:    m = next(momentum_gen)
            except StopIteration: m = EMA_END
            ema_update(target_encoder, context_encoder, m)
            ema_update(target_readout, predictor,        m)

            global_step += 1
            if global_step == WARMUP_STEPS:
                _set_encoder_trainable(True)
                print(f"  [step {global_step}] predictor warmup complete — context_encoder unfrozen.")

            epoch_loss += loss.item(); steps += 1

            if do_diag:
                line = _fmt_diag(diag, global_step, epoch, scheduler.get_last_lr()[0], m, warmup_active)
                warn = _collapse_warn(diag)
                print(line + warn)
                history["diag_steps"].append(global_step)
                history["diag_log"].append(diag)

        avg_loss = epoch_loss / max(steps, 1)
        history["train_loss"].append(avg_loss)
        history["momentum"].append(m)

        improved = avg_loss < best_loss
        if improved:
            best_loss = avg_loss
            save_jepa_ckpt(JEPA_BEST, epoch=epoch, train_loss=avg_loss,
                           best_loss=best_loss, momentum=m, history=history,
                           global_step=global_step)
        save_jepa_ckpt(JEPA_LAST, epoch=epoch, train_loss=avg_loss,
                       best_loss=best_loss, momentum=m, history=history,
                       global_step=global_step)

        tag = "  ★ new best (saved jepa_best.pt)" if improved else ""
        print(f"=== Epoch {epoch:02d}/{EPOCHS} done in {time.time()-t_epoch:.1f}s  "
              f"avg_loss={avg_loss:.4f}  m={m:.4f}{tag}")

    print("\nDone.")

except KeyboardInterrupt:
    print(f"\nInterrupted. Last completed epoch: {epoch}. "
          f"Checkpoints: {JEPA_LAST} (epoch {epoch}), {JEPA_BEST} (best so far).")


In [ ]:
# ---- Diagnostics plot: collapse-trackers over training steps ----
if history.get("diag_log"):
    steps   = history["diag_steps"]
    keys    = ["loss_jepa", "std_b_hpred", "std_t_gctx", "sp_var_ctx", "sp_var_tgt", "cos_mean"]
    labels  = ["JEPA loss", "σ_batch(h_pred)  [collapse↓]", "σ_token(g_ctx)  [collapse↓]",
               "var(soft_pos_ctx)  [collapse↓]", "var(soft_pos_tgt)  [collapse↓]", "cos(h_pred, h_tgt)"]
    fig, axes = plt.subplots(2, 3, figsize=(15, 7))
    for ax, k, lab in zip(axes.flat, keys, labels):
        vals = [d[k] for d in history["diag_log"]]
        ax.plot(steps, vals)
        ax.set_title(lab); ax.set_xlabel("step"); ax.grid(alpha=0.3)
        if "loss" in k: ax.set_yscale("log")
    plt.tight_layout(); plt.show()
else:
    print("No diagnostic log to plot yet — run the training cell first.")


## Linear probe (test-time = 20% only)

In [ ]:
class LinearProbe(nn.Module):
    def __init__(self, in_dim=EMBED_DIM, n_classes=10):
        super().__init__()
        self.fc = nn.Linear(in_dim, n_classes)
    def forward(self, z):
        return self.fc(z)


@torch.no_grad()
def extract_z(pixels, coords):
    context_encoder.eval(); fourier_encoder.eval()
    g, _ = context_encoder(encode_input(pixels, coords), None)
    return g.mean(dim=1)


def train_linear_probe(epochs=10, lr=1e-3, weight_decay=1e-4):
    probe = LinearProbe(in_dim=EMBED_DIM, n_classes=10).to(DEVICE)
    opt   = AdamW(probe.parameters(), lr=lr, weight_decay=weight_decay)
    ce    = nn.CrossEntropyLoss()

    for p in context_encoder.parameters(): p.requires_grad_(False)
    for p in fourier_encoder.parameters(): p.requires_grad_(False)

    start_epoch = 1
    best_acc    = -1.0
    if PROBE_RESUME and os.path.exists(PROBE_LAST):
        s = load_probe_ckpt(PROBE_LAST, probe)
        start_epoch = s["epoch"] + 1
        best_acc    = s.get("best_acc", s.get("test_acc", -1.0))
        if os.path.exists(PROBE_BEST):
            b = torch.load(PROBE_BEST, map_location="cpu", weights_only=False)
            best_acc = max(best_acc, b.get("test_acc", -1.0))
        print(f"Resumed probe from {PROBE_LAST} @ epoch {s['epoch']}  best so far: {best_acc:.4f}")

    @torch.no_grad()
    def _eval():
        probe.eval()
        correct = total = 0
        for b in test_loader:
            ctx_p = b['ctx_pixels'].to(DEVICE)
            ctx_c = b['ctx_coords'].to(DEVICE)
            y     = b['label'].to(DEVICE)
            z     = extract_z(ctx_p, ctx_c)
            correct += (probe(z).argmax(-1) == y).sum().item()
            total   += y.size(0)
        return correct / total

    epoch = start_epoch - 1
    try:
        for epoch in range(start_epoch, epochs + 1):
            probe.train()
            run_loss = correct = total = 0
            for b in train_loader:
                ctx_p = b['ctx_pixels'].to(DEVICE)
                ctx_c = b['ctx_coords'].to(DEVICE)
                y     = b['label'].to(DEVICE)
                z     = extract_z(ctx_p, ctx_c)
                logits = probe(z); loss = ce(logits, y)
                opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
                run_loss += loss.item() * y.size(0)
                correct  += (logits.argmax(-1) == y).sum().item()
                total    += y.size(0)

            train_acc = correct / total
            test_acc  = _eval()
            improved  = test_acc > best_acc
            if improved:
                best_acc = test_acc
                save_probe_ckpt(PROBE_BEST, probe=probe, epoch=epoch,
                                train_acc=train_acc, test_acc=test_acc, best_acc=best_acc)
            save_probe_ckpt(PROBE_LAST, probe=probe, epoch=epoch,
                            train_acc=train_acc, test_acc=test_acc, best_acc=best_acc)
            tag = "  ★ new best (saved probe_best.pt)" if improved else ""
            print(f"[probe ep {epoch:02d}/{epochs}] loss={run_loss/total:.4f}  "
                  f"train_acc={train_acc:.4f}  test_acc(20%)={test_acc:.4f}{tag}")
    except KeyboardInterrupt:
        print(f"\nProbe training interrupted. Last completed epoch: {epoch}. "
              f"Checkpoints: {PROBE_LAST}, {PROBE_BEST}.")
    return probe


probe = train_linear_probe(epochs=10, lr=1e-3)
